In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import matplotlib.pyplot as plt
import networkx as nx

# 定義與原始模型相同的GNN架構
class GNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
    
    def forward(self, x, edge_index, edge_weight=None):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index, edge_weight)
        return x

# 讀取原始數據以獲取節點信息
def load_model_and_data():
    print("讀取保存的模型和數據...")
    
    # 讀取原始數據
    df = pd.read_csv('838-所有評論原始資料.csv', encoding='utf-8')
    
    # 讀取保存的節點嵌入向量
    node_embeddings = np.load('node_embeddings.npy')
    
    # 重新創建地點之間的關係圖以獲取節點列表
    user_locations = df.groupby('user_id')['gmap_location'].apply(list)
    user_locations = user_locations[user_locations.apply(len) > 1]
    
    G = nx.DiGraph()
    for locations in user_locations:
        for i in range(len(locations) - 1):
            source = locations[i]
            target = locations[i+1]
            if G.has_edge(source, target):
                G[source][target]['weight'] += 1
            else:
                G.add_edge(source, target, weight=1)
    
    nodes = list(G.nodes())
    node_mapping = {node: i for i, node in enumerate(nodes)}
    
    # 初始化模型
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = GNN(in_channels=1, hidden_channels=16, out_channels=8).to(device)
    
    # 加載保存的模型權重
    try:
        model.load_state_dict(torch.load('gnn_model.pt', map_location=device))
        model.eval()  # 設置為評估模式
        print("模型加載成功！")
    except Exception as e:
        print(f"加載模型時出錯: {e}")
    
    return model, df, nodes, node_embeddings, node_mapping, device

# 使用加載的模型進行推薦
def recommend_by_text_loaded(user_input, df, nodes, node_embeddings, node_mapping, top_n=5):
    """
    使用加載的模型和嵌入向量進行基於自然語言的景點推薦
    """
    # 定義關鍵詞映射
    keyword_mapping = {
        "玩水": ["水", "海", "海灘", "游泳", "浮潛", "衝浪", "溫泉", "瀑布", "河", "湖", "潭", "游泳池", "戲水"],
        "登山": ["山", "爬山", "登山", "健行", "步道", "森林", "自然", "風景", "登頂", "郊山", "高山"],
        "文化": ["廟宇", "寺廟", "古蹟", "歷史", "文化", "藝術", "博物館", "展覽", "廟", "文化財", "古城", "老街"],
        "美食": ["美食", "小吃", "夜市", "餐廳", "飲食", "特產", "小吃街", "美味", "餐點", "食物", "特色餐"],
        "購物": ["購物", "商場", "市場", "紀念品", "伴手禮", "夜市", "精品", "買", "商店街", "特產", "賣"],
        "放鬆": ["放鬆", "休閒", "溫泉", "SPA", "度假", "渡假村", "療癒", "舒緩", "舒適", "休息", "避暑"],
        "拍照": ["拍照", "風景", "美景", "景色", "打卡", "景點", "自拍", "攝影", "日落", "日出", "壯麗", "美麗"],
        "親子": ["親子", "兒童", "小孩", "家庭", "遊樂", "遊戲", "互動", "教育", "體驗", "適合小朋友", "適合家庭"],
    }
    
    # 提取輸入中的關鍵字
    input_keywords = []
    for category, keywords in keyword_mapping.items():
        for keyword in keywords:
            if keyword in user_input:
                if not any(k in input_keywords for k in keywords):
                    input_keywords.extend(keywords)
                break
    
    # 如果沒有找到特定關鍵字，提取所有非停用詞作為關鍵字
    if not input_keywords:
        # 簡單的中文停用詞
        stopwords = ["我", "想", "要", "去", "的", "是", "有", "在", "來", "能", "會", "可以", "哪裡", "推薦", "地方"]
        input_keywords = [word for word in user_input if word not in stopwords and len(word.strip()) > 0]
    
    # 去除重複關鍵詞並打印
    input_keywords = list(set(input_keywords))
    print(f"提取的關鍵詞: {', '.join(input_keywords)}")
    
    # 計算每個景點與關鍵詞的匹配度
    location_scores = {}
    for location in nodes:
        # 獲取該景點的所有評論
        location_reviews = df[df['gmap_location'] == location]['translated_comments'].fillna('').values
        
        # 關鍵詞匹配次數
        keyword_count = 0
        for review in location_reviews:
            for keyword in input_keywords:
                if keyword in review:
                    keyword_count += 1
        
        # 關鍵詞密度（考慮評論數量）
        review_count = len(location_reviews)
        if review_count > 0:
            keyword_density = keyword_count / review_count
        else:
            keyword_density = 0
        
        # 獲取景點的嵌入向量
        loc_idx = node_mapping.get(location)
        if loc_idx is not None:
            loc_embedding = node_embeddings[loc_idx]
            
            # 計算與其他景點的平均相似度（作為普及度的指標）
            similarities = []
            for i, emb in enumerate(node_embeddings):
                if i != loc_idx:
                    similarity = np.dot(loc_embedding, emb) / (np.linalg.norm(loc_embedding) * np.linalg.norm(emb) + 1e-8)
                    similarities.append(similarity)
            
            avg_similarity = np.mean(similarities) if similarities else 0
            
            # 獲取評論評分
            avg_rating = df[df['gmap_location'] == location]['score'].mean() if review_count > 0 else 0
            
            # 綜合評分：關鍵詞匹配度(0.5) + 景點普及度(0.2) + 評論評分(0.3)
            combined_score = 0.5 * keyword_density + 0.2 * avg_similarity + 0.3 * (avg_rating / 5.0)
            
            # 將結果存入字典
            location_scores[location] = {
                'score': combined_score,
                'keyword_matches': keyword_count,
                'review_count': review_count,
                'avg_rating': avg_rating,
                'popularity': avg_similarity
            }
    
    # 排序並返回推薦結果
    sorted_locations = sorted(location_scores.items(), key=lambda x: x[1]['score'], reverse=True)
    
    # 為最終結果添加詳細信息
    recommendations = []
    for location, data in sorted_locations[:top_n]:
        recommendations.append({
            'location': location,
            'score': data['score'],
            'keyword_matches': data['keyword_matches'],
            'review_count': data['review_count'], 
            'avg_rating': data['avg_rating'],
            'reason': generate_recommendation_reason(location, input_keywords, data)
        })
    
    return recommendations

# 生成推薦原因的輔助函數
def generate_recommendation_reason(location, keywords, data):
    """生成景點被推薦的原因說明"""
    reason = f"{location}很適合您，因為"
    
    if data['keyword_matches'] > 0:
        keyword_phrase = "、".join(keywords[:3]) if len(keywords) > 3 else "、".join(keywords)
        reason += f"在{data['review_count']}則評論中有{data['keyword_matches']}次提到與「{keyword_phrase}」相關的內容，"
    
    # 添加評分信息
    if data['avg_rating'] >= 4.5:
        reason += f"評價極高（{data['avg_rating']:.1f}/5分），"
    elif data['avg_rating'] >= 4.0:
        reason += f"評價優良（{data['avg_rating']:.1f}/5分），"
    elif data['avg_rating'] >= 3.5:
        reason += f"評價良好（{data['avg_rating']:.1f}/5分），"
    
    # 添加普及度信息
    if data['popularity'] > 0.6:
        reason += "且非常受到其他遊客歡迎。"
    elif data['popularity'] > 0.4:
        reason += "且相當受到遊客喜愛。"
    else:
        reason += "是個不錯的選擇。"
        
    return reason

# 互動式推薦界面（使用加載的模型）
def interactive_recommendation_loaded():
    # 加載模型和數據
    model, df, nodes, node_embeddings, node_mapping, device = load_model_and_data()
    
    # 使用KMeans重新計算聚類（可選，如果需要可視化）
    from sklearn.cluster import KMeans
    kmeans = KMeans(n_clusters=4, random_state=42)
    clusters = kmeans.fit_predict(node_embeddings)
    
    # 使用PCA降維以便可視化
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    node_embeddings_2d = pca.fit_transform(node_embeddings)
    
    print("\n" + "="*50)
    print("        台灣旅遊景點推薦系統 (已加載模型)")
    print("="*50)
    print("\n您可以用自然語言描述您想要的旅遊體驗，系統會為您推薦景點。")
    print("例如：「我想要去玩水」、「適合拍照的地方」或「文化歷史景點」等。")
    print("輸入 'exit' 退出。")
    
    while True:
        user_query = input("\n請輸入您的旅遊偏好: ")
        if user_query.lower() == 'exit':
            print("感謝使用，再見！")
            break
        
        if not user_query.strip():
            print("請輸入有效的查詢內容。")
            continue
        
        print("\n正在為您尋找最適合的景點...")
        recommendations = recommend_by_text_loaded(user_query, df, nodes, node_embeddings, node_mapping, top_n=5)
        
        print("\n為您推薦的景點:")
        for i, rec in enumerate(recommendations, 1):
            print(f"{i}. {rec['location']} (相關度: {rec['score']:.2f})")
            print(f"   {rec['reason']}")
        
        # 可視化推薦結果（可選）
        visualize = input("\n是否要顯示推薦景點在聚類空間中的位置? (y/n): ")
        if visualize.lower() == 'y':
            plt.figure(figsize=(14, 10))
            colors = ['#4DBEEE', '#A2142F', '#77AC30', '#7E2F8E']
            markers = ['o', 's', '^', 'd']
            
            # 繪製所有景點（淡化顯示）
            for i in range(len(nodes)):
                plt.scatter(
                    node_embeddings_2d[i, 0], 
                    node_embeddings_2d[i, 1],
                    s=100, 
                    c='lightgray', 
                    alpha=0.3
                )
            
            # 突出顯示推薦景點
            for i, rec in enumerate(recommendations):
                loc = rec['location']
                idx = node_mapping[loc]
                cluster_id = clusters[idx]
                plt.scatter(
                    node_embeddings_2d[idx, 0], 
                    node_embeddings_2d[idx, 1],
                    s=300, 
                    c=colors[cluster_id], 
                    alpha=1.0,
                    marker=markers[cluster_id],
                    edgecolor='black',
                    linewidth=2
                )
                plt.annotate(
                    f"{i+1}. {loc}",
                    (node_embeddings_2d[idx, 0], node_embeddings_2d[idx, 1]),
                    fontsize=14,
                    ha='center', va='center',
                    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", alpha=0.9)
                )
            
            plt.title(f'您的查詢: "{user_query}" 的推薦景點', fontsize=18, pad=20)
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.show()

# 主函數 - 運行互動式推薦界面
if __name__ == "__main__":
    interactive_recommendation_loaded()
